# Summarize the Statements by Year and Reduce Them to the Policy Summary

| Item | Description |
|---|---|
| Goal | Write the policy summary of consumer data privacy legislation with the map-reduce pattern of T1. One call per hearing year summarizes that year's statements (map); one further call combines the 26 yearly summaries (reduce). |
| Input | `policy_summary_input.json` from T3.2.2 in the T3.2 Interim folder: the prompts, the settings, and the numbered statements by year. |
| Output | In the T3.2 Processed folder: `yearly_summaries.csv` (one row per hearing year), `policy_summary.md` (the final text), and `policy_summary_meta.json` (the run record). Every call, with its prompt and completion, in `policy_summary_calls.jsonl` in the Interim folder. |
| Model | `Qwen/Qwen3.5-4B` in bfloat16 on one GPU; `do_sample=True`, `temperature=0.1`, seed 0, reasoning traces off, at most 512 new tokens per call. A completion that hits the cap is cut back to its last full sentence. |
| Length rule | The final summary must have at most 512 bge-large-en-v1.5 tokens. If the reduce output is longer, the call is repeated with a shorter word target, up to three times. |
| Run | Ran on 2026-10-25 on one NVIDIA A40: 26 map calls and 1 reduce call, 176 seconds in all, no call at the cap. The final summary has 221 words and 274 encoder tokens. Not rerun, because sampling would write a new text; Step 9 reads the stored run back. |


## Step 1. Import packages


In [ ]:
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import Markdown, display


## Step 2. Define file locations and settings


In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.2 - Topic Side - Policy Summary of the Legislator Statements")
NOTEBOOK_NAME = "T3.2.3 - Production - Summarize by Year and Reduce to the Policy Summary"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
INPUT_FILE = INTERIM_FOLDER / "policy_summary_input.json"
LOG_FILE = INTERIM_FOLDER / "policy_summary_calls.jsonl"
ENCODER = "BAAI/bge-large-en-v1.5"  # its tokenizer counts the tokens of each summary against the 512 limit

SEED = 0  # sampling at temperature 0.1 is near-greedy; the seed makes a rerun repeat on the same card
SHORTER_TARGET_WORDS = ["150 to 200", "120 to 150", "100 to 120"]  # re-asks if the final summary exceeds the encoder limit
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Point the Hugging Face cache at the scratch root


In [ ]:
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 4. Read the input file

Everything the calls depend on comes from T3.2.2; nothing is retyped here.


In [ ]:
policy_input = json.loads(INPUT_FILE.read_text(encoding="utf-8"))
settings, prompts, chunks, statement_lines = policy_input["settings"], policy_input["prompts"], policy_input["chunks"], policy_input["statement_lines"]
assert policy_input["mode"] == "map_by_year_then_reduce"
assert sum(chunk["n_statements"] for chunk in chunks) == len(statement_lines)
print(f"Statements: {len(statement_lines):,}; map calls: {len(chunks)}; settings: {settings}")


## Step 5. Load the language model and the encoder's tokenizer

The model runs in bfloat16 on the GPU. The encoder's tokenizer only counts tokens.


In [ ]:
assert torch.cuda.is_available(), "run this notebook on a GPU node"
torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(settings["model"])
model = AutoModelForCausalLM.from_pretrained(settings["model"], dtype=torch.bfloat16, device_map="cuda").eval()
bge_tokenizer = AutoTokenizer.from_pretrained(ENCODER)
gpu_name = torch.cuda.get_device_name(0)
run_started = datetime.now(timezone.utc).isoformat(timespec="seconds")
log_handle = LOG_FILE.open("a", encoding="utf-8")
print(f"Model: {settings['model']} on {gpu_name}, {next(model.parameters()).dtype}; run started {run_started}")


## Step 6. Define one call

One prompt, one completion, a fresh context each time. The call is logged with its prompt, its completion, and its token counts before it returns.


In [ ]:
def generate_once(stage, label, user_prompt):
    """Return the completion text, whether it hit the new-token cap, and the token counts."""
    messages = [{"role": "system", "content": prompts["system"]}, {"role": "user", "content": user_prompt}]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=settings["enable_thinking"])
    encoded = tokenizer(rendered, return_tensors="pt").to(model.device)
    n_input_tokens = int(encoded["input_ids"].shape[-1])
    assert n_input_tokens <= settings["input_token_budget"], f"{label}: prompt exceeds the budget"
    started = time.time()
    with torch.no_grad():
        output_ids = model.generate(**encoded, max_new_tokens=settings["max_new_tokens"], pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id, **settings["generation"])
    generated_ids = output_ids[0][n_input_tokens:]
    n_output_tokens = int(len(generated_ids))
    hit_cap = n_output_tokens >= settings["max_new_tokens"]
    text = re.sub(r"^<think>.*?</think>\s*", "", tokenizer.decode(generated_ids, skip_special_tokens=True).strip(), flags=re.DOTALL).strip()
    seconds = time.time() - started
    log_handle.write(json.dumps({"run_started": run_started, "stage": stage, "label": label, "model": settings["model"], "gpu": gpu_name,
                                 "generation": settings["generation"], "enable_thinking": settings["enable_thinking"], "max_new_tokens": settings["max_new_tokens"],
                                 "seed": SEED, "n_input_tokens": n_input_tokens, "n_output_tokens": n_output_tokens, "hit_cap": hit_cap,
                                 "seconds": round(seconds, 1), "user_prompt": user_prompt, "output": text}, ensure_ascii=False) + "\n")
    log_handle.flush()
    print(f"  {stage} {label}: {n_input_tokens:,} in, {n_output_tokens} out, {seconds:.0f}s{', HIT CAP' if hit_cap else ''}", flush=True)
    return text, hit_cap, n_input_tokens, n_output_tokens


def trim_to_sentence(text):
    """Cut a capped completion back to its last complete sentence."""
    ends = [text.rfind(mark) for mark in (". ", "! ", "? ")] + [len(text) if text.endswith((".", "!", "?")) else -1]
    cut = max(ends)
    return text[:cut + 1].strip() if cut > 0 else text


## Step 7. Map: one call per hearing year

Each year's statements go into the map prompt with the year's own target length. A capped completion is trimmed to its last full sentence.


In [ ]:
yearly_rows = []
for chunk in chunks:
    statements_block = "\n".join(statement_lines[number - 1] for number in chunk["statement_numbers"])
    user_prompt = prompts["map_template"].format(n_statements=chunk["n_statements"], year=chunk["hearing_year"], topic_label=policy_input["topic_label"],
                                                 statements_block=statements_block, target_words=settings["map_target_words"])
    summary, hit_cap, n_in, n_out = generate_once("map", str(chunk["hearing_year"]), user_prompt)
    if hit_cap:
        summary = trim_to_sentence(summary)
    yearly_rows.append({"chunk_id": chunk["chunk_id"], "hearing_year": chunk["hearing_year"], "n_statements": chunk["n_statements"], "summary": summary,
                        "n_words": len(summary.split()), "n_bge_tokens": len(bge_tokenizer(summary)["input_ids"]), "hit_cap": hit_cap,
                        "n_input_tokens": n_in, "n_output_tokens": n_out})
yearly_summaries = pd.DataFrame(yearly_rows)
assert yearly_summaries["summary"].str.strip().ne("").all()
yearly_summaries.to_csv(PROCESSED_FOLDER / "yearly_summaries.csv", index=False)
print(f"Saved yearly_summaries.csv: {len(yearly_summaries)} rows")


## Step 8. Reduce: one call over the yearly summaries, then save

The reduce prompt presents the yearly summaries marked with their years. If the result exceeds the encoder limit, the call is repeated with a shorter target. The final text and the run record are saved.


In [ ]:
summaries_block = "\n".join(f"[{row.hearing_year}] {row.summary}" for row in yearly_summaries.itertuples(index=False))
reduce_rounds = []
for round_number, target_words in enumerate([settings["reduce_target_words"]] + SHORTER_TARGET_WORDS, start=1):
    user_prompt = prompts["reduce_template"].format(n_years=len(yearly_summaries), year_min=policy_input["year_min"], year_max=policy_input["year_max"],
                                                    topic_label=policy_input["topic_label"], summaries_block=summaries_block, target_words=target_words)
    summary, hit_cap, n_in, n_out = generate_once("reduce", f"round {round_number} ({target_words} words)", user_prompt)
    if hit_cap:
        summary = trim_to_sentence(summary)
    n_bge_tokens = len(bge_tokenizer(summary)["input_ids"])
    reduce_rounds.append({"round": round_number, "target_words": target_words, "n_words": len(summary.split()), "n_bge_tokens": n_bge_tokens, "hit_cap": hit_cap})
    if n_bge_tokens <= settings["max_bge_tokens"]:
        break
else:
    raise RuntimeError(f"final summary still exceeds {settings['max_bge_tokens']} encoder tokens after {len(reduce_rounds)} rounds")
log_handle.close()

(PROCESSED_FOLDER / "policy_summary.md").write_text(summary + "\n", encoding="utf-8")
(PROCESSED_FOLDER / "policy_summary_meta.json").write_text(json.dumps({
    "topic_id": policy_input["topic_id"], "topic_label": policy_input["topic_label"], "n_statements": len(statement_lines), "n_map_calls": len(chunks),
    "reduce_rounds": reduce_rounds, "final_n_words": len(summary.split()), "final_n_bge_tokens": n_bge_tokens, "model": settings["model"], "gpu": gpu_name,
    "generation": settings["generation"], "enable_thinking": settings["enable_thinking"], "max_new_tokens": settings["max_new_tokens"], "seed": SEED,
    "run_started": run_started, "run_finished": datetime.now(timezone.utc).isoformat(timespec="seconds"), "call_log": LOG_FILE.name}, indent=2), encoding="utf-8")
print(f"Final summary: {len(summary.split())} words, {n_bge_tokens} encoder tokens, after {len(reduce_rounds)} reduce round(s)")


## Step 9. Read the stored run back

Evidence of the October run: the run record, the call table from the log, and the final policy summary. This cell runs without the model.


In [3]:
meta = json.loads((PROCESSED_FOLDER / "policy_summary_meta.json").read_text(encoding="utf-8"))
calls = pd.read_json(LOG_FILE, lines=True)
calls = calls.loc[calls["run_started"].eq(meta["run_started"])]
print(f"Run {meta['run_started']} to {meta['run_finished']} on {meta['gpu']}; model {Path(meta['model']).name}; generation {meta['generation']}, thinking {meta['enable_thinking']}, seed {meta['seed']}")
print(f"Calls: {len(calls)} ({int(calls['stage'].eq('map').sum())} map, {int(calls['stage'].eq('reduce').sum())} reduce); {calls['seconds'].sum():.0f} seconds; at the cap: {int(calls['hit_cap'].sum())}")
print(f"Final summary: {meta['final_n_words']} words, {meta['final_n_bge_tokens']} encoder tokens")
display(calls[["stage", "label", "n_input_tokens", "n_output_tokens", "seconds"]].reset_index(drop=True))
display(Markdown("**Policy summary, Topic 68**\n\n" + (PROCESSED_FOLDER / "policy_summary.md").read_text(encoding="utf-8")))


Run 2026-10-25T12:37:24+00:00 to 2026-10-25T12:40:20+00:00 on NVIDIA A40; model 851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a; generation {'do_sample': True, 'temperature': 0.1}, thinking False, seed 0
Calls: 27 (26 map, 1 reduce); 176 seconds; at the cap: 0
Final summary: 221 words, 274 encoder tokens


,stage,label,n_input_tokens,n_output_tokens,seconds
0,map,1999,683,164,8.9
1,map,2000,1666,118,5.3
2,map,2001,1100,171,7.4
3,map,2002,1350,135,6.0
4,map,2003,648,122,5.3
5,map,2004,547,132,5.7
6,map,2005,346,111,4.8
7,map,2006,606,128,5.5
8,map,2007,281,114,4.9
9,map,2008,612,141,6.0


**Policy summary, Topic 68**

The legislation imposes rigorous obligations on tech giants, broadband providers, healthcare organizations, drone operators, and financial institutions to secure consumer data and prevent its unauthorized sale or misuse. It grants individuals specific rights, including the ability to opt in or out of data collection, demand clear and conspicuous privacy policies, control their digital communications, and establish data "firewalls." Companies are required to implement "privacy by design," provide breach notifications within 72 hours, and ensure that sensitive information such as health records, social security numbers, and geolocation data is protected from discrimination and identity theft.

The regulatory scope targets a wide array of data practices, ranging from the deceptive monitoring of online activities and the unauthorized mapping of consumer movements to the commercialization of private data and the use of algorithms to amplify or suppress content. Specific actors include research centers, third-party aggregators, and federal agencies, with particular emphasis on preventing the export of products and the sharing of personal information without consent. The framework addresses diverse data types, including directory information, voice recordings, and data gathered by connected vehicles, aiming to balance national security interests with civil liberties. Ultimately, the legislation seeks to replace fragmented standards with a uniform national baseline that mandates transparency, enforces accountability, and empowers consumers to make informed choices regarding their personal information in the digital economy.
